In [16]:
import requests
import time

DAY_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson"
HOUR_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_hour.geojson"


def get_earthquake(url):
    response = requests.get(url)
    data = response.json()
    return data


data = get_earthquake(DAY_URL)
data.keys()

dict_keys(['type', 'metadata', 'features', 'bbox'])

In [17]:
for feature in data["features"]:
    print(feature["properties"])
    break

{'mag': 1.04, 'place': '2 km NW of The Geysers, CA', 'time': 1774195159780, 'updated': 1774195254475, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75331457', 'detail': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/nc75331457.geojson', 'felt': None, 'cdi': None, 'mmi': None, 'alert': None, 'status': 'automatic', 'tsunami': 0, 'sig': 17, 'net': 'nc', 'code': '75331457', 'ids': ',nc75331457,', 'sources': ',nc,', 'types': ',nearby-cities,origin,phase-data,', 'nst': 15, 'dmin': 0.01651, 'rms': 0.03, 'gap': 128, 'magType': 'md', 'type': 'earthquake', 'title': 'M 1.0 - 2 km NW of The Geysers, CA'}


| Field | Example Value | Description |
|---|---|---|
| `mag` | `1.11` | Earthquake magnitude |
| `place` | `"7 km NW of The Geysers, CA"` | Human-readable location description |
| `time` | `1774143895690` | Unix timestamp in milliseconds — divide by 1000 to convert |
| `type` | `"earthquake"` | Event type: `earthquake`, `quarry blast`, `explosion` |
| `tsunami` | `0` | Tsunami alert — `0` no, `1` yes |
| `sig` | `19` | Significance score 0–1000 — combines magnitude, depth, and nearby population |
| `magType` | `"md"` | Magnitude calculation method: `ml`, `md`, `mw`, etc. |
| `title` | `"M 1.1 - 7 km NW..."` | Human-readable summary — useful for tooltips |
| `status` | `"automatic"` | `automatic` = algorithm-generated, `reviewed` = verified by a human |
| `updated` | `1774143993933` | Last update timestamp from USGS in milliseconds |
| `felt` | `None` | Number of people who reported feeling it |
| `cdi` | `None` | Community Decimal Intensity — crowd-sourced shaking intensity |
| `mmi` | `None` | Maximum estimated intensity (Modified Mercalli Intensity) |
| `alert` | `None` | PAGER alert level: `green`, `yellow`, `orange`, `red` |
| `net` | `"nc"` | Detecting seismic network: `ak`=Alaska, `nc`=N. California, `ci`=S. California |
| `nst` | `15` | Number of sensors used to calculate the location |
| `dmin` | `0.002042` | Distance to nearest sensor in degrees |
| `rms` | `0.02` | Model fit error — lower is more precise |
| `gap` | `93` | Largest azimuthal gap without sensor coverage — larger gap means less precise location |
| `lat` | `-38.7312` | Latitude of the epicenter in decimal degrees |
| `lon` | `-122.7891` | Longitude of the epicenter in decimal degrees |
| `depth` | `2.79` | Depth of the hypocenter in kilometers below the surface |
| `url` | `"https://..."` | Link to the event page on USGS |
| `detail` | `"https://..."` | URL to the event's detailed GeoJSON |
| `tz` | `None` | Timezone offset — always `None` in the modern feed |
| `ids` | `",nc75331247,"` | All IDs assigned by contributing networks |
| `sources` | `",nc,"` | Networks that contributed data |
| `types` | `",origin,phase-data,"` | Data product types available for this event |
| `code` | `"75331247"` | Internal ID from the detecting network |

In [18]:
from dataclasses import dataclass


@dataclass
class Earthquake:
    mag: float
    mag_type: str
    place: str
    tsunami: int
    event_time: int
    event_type: str
    title: str
    sig: int
    lat: float
    lon: float
    depth: float


def safe_int(val, default=0):
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


def safe_float(val, default=0.0):
    try:
        return float(val)
    except (ValueError, TypeError):
        return default


def earthquake_from_serie(serie):
    return Earthquake(
        mag=safe_float(serie["mag"]),
        mag_type=str(serie["mag_type"]),
        place=str(serie["place"]),
        tsunami=safe_int(serie["tsunami"]),
        event_time=safe_int(serie["event_time"]),
        event_type=str(serie["event_type"]),
        title=str(serie["title"]),
        sig=safe_int(serie["sig"]),
        lat=safe_float(serie["lat"]),
        lon=safe_float(serie["lon"]),
        depth=safe_float(serie["depth"]),
    )

In [20]:
import pandas as pd
import dataclasses
import json
from kafka import KafkaProducer


def dict2series(features):
    props = features["properties"]
    coords = features["geometry"]["coordinates"]

    serie = pd.Series(
        {
            "mag": props["mag"],
            "mag_type": props["magType"],
            "place": props["place"],
            "tsunami": props["tsunami"],
            "event_time": props["time"],
            "event_type": props["type"],
            "title": props["title"],
            "sig": props["sig"],
            "lon": coords[0],  # ojo el orden
            "lat": coords[1],
            "depth": coords[2],
        }
    )
    return serie


def earthquake_serializer(earthquake):
    earthquake_dict = dataclasses.asdict(earthquake)
    json_str = json.dumps(earthquake_dict)
    return json_str.encode("utf-8")


INTERVAL = 60
server = "localhost:9092"
topic_name = "earthquake-feeds"

producer = KafkaProducer(
    bootstrap_servers=[server], value_serializer=earthquake_serializer
)

In [21]:
seen = set()

# Carga inicial - ultimas 24hs
data = get_earthquake(DAY_URL)

count = 0
for features in data["features"]:
    if features["id"] not in seen:
        seen.add(features["id"])
        serie = dict2series(features)
        earthquake = earthquake_from_serie(serie)
        producer.send(topic=topic_name, value=earthquake)
        count += 1

print(f"{count} earthquakes this day")
producer.flush()

seen = set()
count = 0
# Loop incremental - solo novedades
while True:
    data = get_earthquake(HOUR_URL)

    for features in data["features"]:
        if features["id"] not in seen:
            seen.add(features["id"])
            serie = dict2series(features)
            earthquake = earthquake_from_serie(serie)
            producer.send(topic=topic_name, value=earthquake)
            count += 1

    print(f"{count} earthquakes this hour")
    producer.flush()
    time.sleep(INTERVAL)

249 earthquakes this day
9 earthquakes this hour
9 earthquakes this hour
9 earthquakes this hour
9 earthquakes this hour
9 earthquakes this hour


KeyboardInterrupt: 

In [10]:
data = get_earthquake(DAY_URL)

for features in data["features"]:
    print(features)

{'type': 'Feature', 'properties': {'mag': 1.8, 'place': '63 km ESE of Denali National Park, Alaska', 'time': 1774153840909, 'updated': 1774154055492, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/aka2026frfbre', 'detail': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/aka2026frfbre.geojson', 'felt': None, 'cdi': None, 'mmi': None, 'alert': None, 'status': 'automatic', 'tsunami': 0, 'sig': 50, 'net': 'ak', 'code': 'a2026frfbre', 'ids': ',aka2026frfbre,', 'sources': ',ak,', 'types': ',origin,phase-data,', 'nst': 21, 'dmin': 0.7, 'rms': 0.5, 'gap': 99, 'magType': 'ml', 'type': 'earthquake', 'title': 'M 1.8 - 63 km ESE of Denali National Park, Alaska'}, 'geometry': {'type': 'Point', 'coordinates': [-150.629, 63.256, 129.6]}, 'id': 'aka2026frfbre'}
{'type': 'Feature', 'properties': {'mag': 1.52, 'place': '75 km ENE of Tonopah, Nevada', 'time': 1774153797584, 'updated': 1774153975098, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nn0